https://www.kci.go.kr/kciportal/ci/sereArticleSearch/ciSereArtiView.kci?sereArticleSearchBean.artiId=ART003153102

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

**Warning**

Since I used Korean, I encoded it as cp949 instead of utf-8

In [ ]:
df = pd.read_csv("NewsResult_20220127-20241231.csv", encoding='cp949')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10948 entries, 0 to 10947
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   뉴스 식별자             10948 non-null  float64
 1   일자                 10948 non-null  int64  
 2   언론사                10948 non-null  object 
 3   기고자                10502 non-null  object 
 4   제목                 10948 non-null  object 
 5   통합 분류1             10948 non-null  object 
 6   통합 분류2             10413 non-null  object 
 7   통합 분류3             9204 non-null   object 
 8   사건/사고 분류1          652 non-null    object 
 9   사건/사고 분류2          83 non-null     object 
 10  사건/사고 분류3          20 non-null     object 
 11  인물                 7199 non-null   object 
 12  위치                 9643 non-null   object 
 13  기관                 10942 non-null  object 
 14  키워드                10948 non-null  object 
 15  특성추출(가중치순 상위 50개)  10948 non-null  object 
 16  본문                 109

In [ ]:
columns_to_drop = [
    '사건/사고 분류1',
    '사건/사고 분류2',
    '사건/사고 분류3',
    '위치',
    '분석제외 여부',
    '통합 분류1',
    '통합 분류2',
    '통합 분류3',
    '인물',
    '기고자',
    '뉴스 식별자',
    '언론사',
    'URL',
    '기관',
    '특성추출(가중치순 상위 50개)',
]

In [ ]:
df.drop(columns=columns_to_drop, inplace=True)

In [ ]:
df.head()

,일자,제목,키워드,본문
0,20241231,[주용석 칼럼] '국장 탈출은 지능 순' 조롱 싫다면,"국장,탈출,지능,증시,한국,우울,MSCI,신흥국,지수,한국,비중,중국,대만,인도,4...","2024년 한국 증시는 우울했다. MSCI 신흥국지수에서 한국 비중은 중국, 대만,..."
1,20241231,찬바람 이겨내고 내면을 가득 채울 문장 전합니다,"찬바람,내면,문장,겨울바람,마음,겨울,선택,선택,영국,소설가,캐서린,메이,인생,겨울...",매섭게 부는 겨울바람처럼 어수선하고 무거운 마음으로 보낸 한 해였습니다. “우리는 ...
2,20241231,"삼성, 레인보우로보틱스 최대주주로 '인간형 로봇 전쟁' 참전","삼성,레인보우로보틱스,최대,주주,인간형,로봇,전쟁,참전,삼성전자,휴머노이드,로봇,사...",삼성전자가 휴머노이드 등 로봇 사업을 새로운 성장동력으로 선정하고 회사 역량을 집중...
3,20241231,"""단체문자 카톡 보내려면"" 은행 백화점 공공기관도 '이곳' 쓴다","단체문자,카톡,은행,백화점,공공기관,티젠소프트,사람,사이,소통,핵심,가치,멀티미디어...","""티젠소프트는 '사람들 사이의 소통'을 핵심 가치로 삼아 이를 '멀티미디어 솔루션'..."
4,20241231,전력 K푸드가 이끌다 정치 테마株 '블랙홀'로 [2024 증시 결산],"전력,K푸드,정치,테마,블랙홀,결산,증시,2024년,증시,수익,동향,상반기,상반기,...",2024년 한해 국내 증시 수익률 동향은 상반기와 하반기가 극명히 엇갈렸다. 상반기...


In [ ]:
df.isna().sum()

,0
일자,0
제목,0
키워드,0
본문,0


https://www.kci.go.kr/kciportal/ci/sereArticleSearch/ciSereArtiView.kci?sereArticleSearchBean.artiId=ART003055323

#KoBERT

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

In [ ]:
df['full_text'] = df['제목'] + " " + df['키워드'] + " " + df['본문']

In [ ]:
train_texts = df['full_text'].tolist()
train_labels = [2, 1, 0]

In [ ]:
MODEL_NAME = "monologg/kobert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

The repository monologg/kobert contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/monologg/kobert .
 You can inspect the repository content at https://hf.co/monologg/kobert.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at monologg/kobert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
class NewsSentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True)

In [ ]:
train_dataset = NewsSentimentDataset(train_encodings, train_labels)